# Notebook to Compare Heuristics

In [1]:
import random
import networkx as nx

from time import time

random.seed(42)  # For accurate comparison

In [2]:
from src import (
    kempe_greedy,
    welfare_greedy,
)

from src import (
    estimate_influence,
    independent_cascade_community
)

## Comparison of Kempe Greedy and Welfare Greedy

In [3]:
graph = nx.erdos_renyi_graph(
    n=100,
    p=0.05,
    seed=42,
    directed=True,
)

# Assign communities randomly
for i, node in enumerate(graph.nodes()):
    graph.nodes[node]['community'] = random.randint(0, 2)

communities = set(nx.get_node_attributes(graph, 'community').values())

In [4]:
k = 5  # number of seeds to select
alpha = 1.5  # inequality-aversion parameter (higher = more fairness)
p = 0.1  # edge activation probability
num_sims = 1000

In [5]:
start = time()
kempe_seeds = kempe_greedy(
    graph=graph,
    k=k,
    probability=p,
    num_simulations=num_sims,
)
kempe_time = time() - start

kempe_influence = estimate_influence(
    graph=graph,
    seeds=kempe_seeds,
    propagation_prob=p,
    num_simulations=num_sims,
)

kempe_by_comm = independent_cascade_community(
    graph=graph,
    seeds=kempe_seeds,
    probability=p,
    num_sims=500,
)

Selecting seeds: 100%|██████████| 5/5 [00:03<00:00,  1.29it/s]


In [6]:
start = time()
welfare_seeds = welfare_greedy(
    graph=graph,
    communities=communities,
    k=k,
    alpha=alpha,
    probability=p,
    num_sims=num_sims,
)
welfare_time = time() - start

welfare_influence = estimate_influence(
    graph=graph,
    seeds=welfare_seeds,
    propagation_prob=p,
    num_simulations=num_sims,
)

welfare_by_comm = independent_cascade_community(
    graph=graph,
    seeds=welfare_seeds,
    probability=p,
    num_sims=500,
)

Selecting seeds: 100%|██████████| 5/5 [00:02<00:00,  2.04it/s]


In [7]:
print(f'Kempe Greedy:\n Seeds: {kempe_seeds}\n Time: {kempe_time:.2f}s\n Average Influence: {kempe_influence:.1f}\n')
print(f'Welfare Greedy:\n Seeds: {welfare_seeds}\n Time: {welfare_time:.2f}s\n Average Influence: {welfare_influence:.1f}\n')

Kempe Greedy:
 Seeds: {14, 47, 50, 21, 56}
 Time: 3.90s
 Average Influence: 12.4

Welfare Greedy:
 Seeds: {34, 72, 75, 14, 93}
 Time: 2.45s
 Average Influence: 10.1



In [8]:
print(f'Average Influence by Community (Kempe): {kempe_by_comm}')
print(f'Average Influence by Community (Welfare): {welfare_by_comm}')

Average Influence by Community (Kempe): {0: 0.41471030233007183, 1: 0.15275369431536176, 2: 0.4325360033545673}
Average Influence by Community (Welfare): {0: 0.35103187887104303, 1: 0.31915853755807344, 2: 0.3298095835708847}
